# Practice Archetypes — UK Map

This notebook geocodes each practice's postcode to a latitude/longitude coordinate
and plots all practices on a UK map, coloured by their archetype label.

Two views are provided:

1. **Interactive scatter map** (Plotly) — hover over any practice to see its name,
   archetype, region and key metrics. Zoom in to any area of the UK.

2. **Static regional summaries** (Matplotlib) — which archetypes dominate each region,
   and how staffing and income vary geographically across the archetype mix.

## Geocoding

Postcodes are resolved to lat/lon using `pgeocode`, which queries the GeoNames
database locally — no API key or internet connection required at runtime after
the first download.

> **Synthetic data note:** The practice postcodes in `master.csv` are all synthetic
> SW London postcodes (e.g. SW19 4AB). On the interactive map, all practices will
> cluster in South-West London. With real postcode data the practices will spread
> correctly across the UK. The regional charts use the `region` column, which is
> unaffected by this.

> Run `01_rules_based.ipynb` first to generate `archetypes_rules.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import pgeocode
import warnings

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

SIZE_ORDER  = ['Small/Foundation', 'Medium/Core', 'Large/Advanced', 'Flagship']
MODEL_ORDER = ['NHS Led', 'Balanced Mixed', 'Private Led Mixed', 'Specialist/Referral Hub']

# Archetype colours — consistent across both views
SIZE_PALETTE  = {'Small/Foundation': '#74add1', 'Medium/Core': '#4575b4',
                 'Large/Advanced': '#d73027', 'Flagship': '#a50026'}
MODEL_PALETTE = {'NHS Led': '#1a9850', 'Balanced Mixed': '#fee08b',
                 'Private Led Mixed': '#f46d43', 'Specialist/Referral Hub': '#9e0142'}

# Load data
master = pd.read_csv('master.csv')
arc    = pd.read_csv('archetypes_rules.csv')[['practicekey','archetype_size','archetype_model','archetype_rules']]
df = master.merge(arc, on='practicekey', how='inner')

# Derived metrics for hover info
df['nhs_income_est']   = df['nhsincome'].fillna(0)
df['private_income']   = df['privateincome'].fillna(0)
df['total_income_est'] = df['private_income'] + df['nhs_income_est']
df['total_headcount']  = df[['position_dentist','position_dental_nurse','position_hygienist',
                              'position_receptionist','position_practice_manager']].sum(axis=1)

print(f'Loaded {len(df):,} practices')
print(f'Regions: {sorted(df["region"].unique())}')

---
## Geocode postcodes

`pgeocode` looks up latitude and longitude from the UK postcode database.
The first run downloads the data file; subsequent runs use a local cache.
Unrecognised postcodes (e.g. truncated or malformed) fall back to the
known centroid for that practice's region.

In [ ]:
# Region centroids — fallback for practices whose postcode cannot be resolved
REGION_CENTROIDS = {
    'London':    (51.51,  -0.13),
    'Midlands':  (52.48,  -1.90),
    'North East':(54.97,  -1.62),
    'Scotland':  (56.49,  -4.20),
    'Wales':     (52.13,  -3.78),
    'South East':(51.45,  -0.97),
    'South West':(50.72,  -3.53),
    'North West':(53.47,  -2.24),
    'Yorkshire': (53.80,  -1.55),
    'East':      (52.24,   0.90),
}

print('Geocoding postcodes with pgeocode …')
nomi = pgeocode.Nominatim('GB')

unique_postcodes = df['postcode'].unique()
geo = nomi.query_postal_code(unique_postcodes.tolist())
geo.index = unique_postcodes
geo_map = geo[['latitude', 'longitude']]

df['lat'] = df['postcode'].map(geo_map['latitude'])
df['lon'] = df['postcode'].map(geo_map['longitude'])

# Apply region centroid fallback for any unresolved postcodes
missing_mask = df['lat'].isna() | df['lon'].isna()
if missing_mask.any():
    df.loc[missing_mask, 'lat'] = df.loc[missing_mask, 'region'].map(
        lambda r: REGION_CENTROIDS.get(r, (54.0, -2.0))[0])
    df.loc[missing_mask, 'lon'] = df.loc[missing_mask, 'region'].map(
        lambda r: REGION_CENTROIDS.get(r, (54.0, -2.0))[1])
    # Add small jitter to prevent exact overlaps on region centroids
    rng = np.random.default_rng(42)
    df.loc[missing_mask, 'lat'] += rng.uniform(-0.3, 0.3, missing_mask.sum())
    df.loc[missing_mask, 'lon'] += rng.uniform(-0.3, 0.3, missing_mask.sum())

resolved = (~missing_mask).sum()
print(f'Geocoded: {resolved:,} / {len(df):,} practices resolved from postcode')
print(f'Fallback to region centroid: {missing_mask.sum():,} practices')
print()
print('Lat/lon sample:')
display(df[['practicename', 'postcode', 'region', 'lat', 'lon']].head(5))

---
## Interactive map — coloured by model archetype

Each dot is a practice. Hover for name, archetype, region, income and headcount.
Use the legend to toggle archetypes on/off. The map uses OpenStreetMap tiles via
Plotly's `open-street-map` style (no API key required).

Switch the `color_col` variable to `archetype_size` to view the size dimension instead.

In [ ]:
color_col   = 'archetype_model'           # swap to 'archetype_size' for size view
color_order = MODEL_ORDER                 # swap to SIZE_ORDER
color_map   = MODEL_PALETTE               # swap to SIZE_PALETTE

df['hover_text'] = (
    df['practicename'] + '<br>' +
    'Region: '     + df['region']          + '<br>' +
    'Size: '       + df['archetype_size'].astype(str)  + '<br>' +
    'Model: '      + df['archetype_model'] + '<br>' +
    'Surgeries: '  + df['numberofsurgeries'].astype(str) + '<br>' +
    'Staff: '      + df['total_headcount'].astype(str)   + '<br>' +
    'Est. income: £' + df['total_income_est'].round(0).astype(int).astype(str)
)

fig = px.scatter_mapbox(
    df,
    lat='lat', lon='lon',
    color=color_col,
    color_discrete_map=color_map,
    category_orders={color_col: color_order},
    hover_name='practicename',
    hover_data={'lat': False, 'lon': False,
                'region': True,
                'archetype_size': True,
                'archetype_model': True,
                'numberofsurgeries': True,
                'total_headcount': True},
    zoom=5,
    center={'lat': 54.5, 'lon': -3.0},
    mapbox_style='open-street-map',
    height=650,
    title='Practice Archetypes — UK Map (Model)',
    opacity=0.85,
    size_max=12,
)
fig.update_traces(marker_size=10)
fig.update_layout(
    legend_title_text='Model archetype',
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

---
## Interactive map — coloured by size archetype

In [ ]:
fig2 = px.scatter_mapbox(
    df,
    lat='lat', lon='lon',
    color='archetype_size',
    color_discrete_map=SIZE_PALETTE,
    category_orders={'archetype_size': SIZE_ORDER},
    hover_name='practicename',
    hover_data={'lat': False, 'lon': False,
                'region': True,
                'archetype_size': True,
                'archetype_model': True,
                'numberofsurgeries': True,
                'total_headcount': True},
    zoom=5,
    center={'lat': 54.5, 'lon': -3.0},
    mapbox_style='open-street-map',
    height=650,
    title='Practice Archetypes — UK Map (Size)',
    opacity=0.85,
)
fig2.update_traces(marker_size=10)
fig2.update_layout(
    legend_title_text='Size archetype',
    margin=dict(l=0, r=0, t=40, b=0),
)
fig2.show()

---
## Regional archetype distribution

The region column is independent of the synthetic postcodes and correctly reflects
the simulated geographic distribution. These charts show the archetype mix within
each region.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Regional Archetype Distribution', fontsize=13, fontweight='bold')

regions = sorted(df['region'].unique())

# Model mix by region (stacked % bar)
model_ct = pd.crosstab(df['region'], df['archetype_model']).reindex(
    index=regions, columns=MODEL_ORDER, fill_value=0)
model_pct = model_ct.div(model_ct.sum(axis=1), axis=0) * 100

bottom = np.zeros(len(regions))
x = np.arange(len(regions))
for model_val in MODEL_ORDER:
    vals = model_pct[model_val].values
    axes[0].bar(x, vals, bottom=bottom, color=MODEL_PALETTE[model_val],
                label=model_val, edgecolor='white')
    for xi, (v, b) in enumerate(zip(vals, bottom)):
        if v > 8:
            axes[0].text(xi, b + v/2, f'{v:.0f}%', ha='center', va='center',
                         fontsize=8, color='black')
    bottom += vals
axes[0].set_xticks(x)
axes[0].set_xticklabels(regions, rotation=20, ha='right')
axes[0].set_ylabel('% of practices in region')
axes[0].set_title('Model archetype mix by region')
axes[0].legend(loc='lower right', fontsize=8)
axes[0].set_ylim(0, 105)

# Size mix by region
size_ct  = pd.crosstab(df['region'], df['archetype_size']).reindex(
    index=regions, columns=SIZE_ORDER, fill_value=0)
size_pct = size_ct.div(size_ct.sum(axis=1), axis=0) * 100

bottom = np.zeros(len(regions))
for size_val in SIZE_ORDER:
    vals = size_pct[size_val].values
    axes[1].bar(x, vals, bottom=bottom, color=SIZE_PALETTE[size_val],
                label=size_val, edgecolor='white')
    for xi, (v, b) in enumerate(zip(vals, bottom)):
        if v > 8:
            axes[1].text(xi, b + v/2, f'{v:.0f}%', ha='center', va='center',
                         fontsize=8, color='white')
    bottom += vals
axes[1].set_xticks(x)
axes[1].set_xticklabels(regions, rotation=20, ha='right')
axes[1].set_ylabel('% of practices in region')
axes[1].set_title('Size archetype mix by region')
axes[1].legend(loc='lower right', fontsize=8)
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.show()

---
## Regional income and staffing by archetype

Do the same archetypes perform similarly across regions, or do regional factors
(e.g. market rates, NHS contract availability, demographics) create meaningful
differences within an archetype?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Income and Staffing by Region and Model Archetype', fontsize=13, fontweight='bold')

region_arch = (
    df.groupby(['region', 'archetype_model'], observed=True)
    .agg(
        n=('practicekey', 'count'),
        avg_total_income=('total_income_est', 'mean'),
        avg_headcount=('total_headcount', 'mean'),
    )
    .reset_index()
)

for ax, metric, ylabel, title in [
    (axes[0], 'avg_total_income', 'Avg total income (£)', 'Average income by region and model'),
    (axes[1], 'avg_headcount',    'Avg headcount',        'Average headcount by region and model'),
]:
    for model_val in MODEL_ORDER:
        sub = region_arch[region_arch['archetype_model'] == model_val].set_index('region')
        sub = sub.reindex(regions)
        ax.plot(regions, sub[metric], marker='o', lw=2,
                color=MODEL_PALETTE[model_val], label=model_val)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticklabels(regions, rotation=20, ha='right')
    ax.legend(fontsize=8)
    if metric == 'avg_total_income':
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'£{v/1e3:.0f}k'))

plt.tight_layout()
plt.show()

---
## Dominant archetype per region — summary table

In [ ]:
dominant = (
    df.groupby(['region', 'archetype_model'], observed=True)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .groupby('region')
    .first()
    .rename(columns={'archetype_model': 'dominant_model', 'count': 'n_practices'})
)

dominant_size = (
    df.groupby(['region', 'archetype_size'], observed=True)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .groupby('region')
    .first()
    .rename(columns={'archetype_size': 'dominant_size'})[['dominant_size']]
)

region_summary = (
    df.groupby('region')
    .agg(
        total_practices=('practicekey', 'count'),
        avg_surgeries=('numberofsurgeries', 'mean'),
        avg_headcount=('total_headcount', 'mean'),
        avg_income=('total_income_est', 'mean'),
    )
    .round(1)
    .join(dominant[['dominant_model']])
    .join(dominant_size)
)

display(region_summary)

---
## Static UK region scatter (Matplotlib)

For contexts where Plotly is not available (e.g. exported PDFs, PowerPoint slides),
this chart uses the actual lat/lon coordinates on a simple scatter without a map
tile background. The rough shape of the UK is recognisable from the practice
distribution alone when real postcode data is used.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 9))
fig.suptitle('Practice Locations — Coloured by Archetype', fontsize=13, fontweight='bold')

for ax, col, palette, order, legend_title in [
    (axes[0], 'archetype_model', MODEL_PALETTE, MODEL_ORDER, 'Model'),
    (axes[1], 'archetype_size',  SIZE_PALETTE,  SIZE_ORDER,  'Size'),
]:
    for arch in order:
        sub = df[df[col] == arch]
        ax.scatter(sub['lon'], sub['lat'],
                   c=palette[arch], label=arch,
                   s=40, alpha=0.7, edgecolors='white', lw=0.4)

    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'Coloured by {legend_title}')
    ax.legend(title=legend_title, fontsize=8, loc='lower left')
    ax.set_aspect('equal')

    # Annotate region centroids
    for region, (clat, clon) in REGION_CENTROIDS.items():
        if region in df['region'].values:
            ax.text(clon, clat, region, fontsize=7, color='black',
                    ha='center', va='bottom',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.6, edgecolor='none'))

plt.tight_layout()
plt.show()